<a href="https://colab.research.google.com/github/ngando2772/Linkedin_Redacteur_Post/blob/main/fine_tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =========================
# 0. IMPORTS UNSLOTH
# =========================
from unsloth import FastLanguageModel
from datasets import load_dataset
from transformers import TrainingArguments

# =========================
# 1. CONFIG
# =========================
MODEL_ID = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"
DATASET_PATH = "/content/bible_style.jsonl"

# =========================
# 2. DATASET
# =========================
dataset = load_dataset("json", data_files=DATASET_PATH, split="train")

print(dataset[0])

# =========================
# 3. LOAD MODEL (ULTRA OPTIMIZED)
# =========================
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_ID,
    max_seq_length = 1024,
    dtype = None,
    load_in_4bit = True,
)

# =========================
# 4. ADD LORA (UNSLOTH STYLE)
# =========================
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_alpha = 32,
    lora_dropout = 0.05,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

# =========================
# 5. FORMAT DATA
# =========================
def format_text(example):
    return {"text": example["text"]}

dataset = dataset.map(format_text)

# =========================
# 6. TRAINING ARGS
# =========================
training_args = TrainingArguments(
    output_dir="./unsloth-qwen-bible",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=5,
    max_steps=200,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    optim="adamw_8bit",
    report_to="none",
)

# =========================
# 7. TRAINER
# =========================
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=1024,
    args=training_args,
)

# =========================
# 8. TRAIN
# =========================
print("🔥 START TRAINING UNSLOTH")
trainer.train()

# =========================
# 9. SAVE MODEL
# =========================
model.save_pretrained("./unsloth-qwen-bible")
tokenizer.save_pretrained("./unsloth-qwen-bible")

print("✅ DONE - FAST + STABLE MODEL SAVED")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


Generating train split: 0 examples [00:00, ? examples/s]

{'text': 'Genesis 1:1 - Dieu, au commencement, créa les cieux et la terre;'}
==((====))==  Unsloth 2026.5.7: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.05G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

...

🔥 START TRAINING UNSLOTH


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 30,975 | Num Epochs = 1 | Total steps = 200
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
10,2.516814
20,2.011409
30,1.953849
40,1.772691
50,1.851727
60,1.783894
70,1.837060
80,1.752302
90,1.743619
100,1.834122


Unsloth: Restored added_tokens_decoder metadata in ./unsloth-qwen-bible/checkpoint-200/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./unsloth-qwen-bible/tokenizer_config.json.


✅ DONE - FAST + STABLE MODEL SAVED


In [ ]:
!pip uninstall -y transformers peft trl bitsandbytes accelerate -q
!pip install -q transformers==4.46.0 peft==0.13.0 accelerate==0.34.2 safetensors sentencepiece

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 2.9 MB/s eta 0:00:00
Reason for being yanked: This version unfortunately does not work with 3.8 but we did not drop the support yet
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 82.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.5/322.5 kB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.4/324.4 kB 34.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unsloth-zoo 2026.5.4 requires trl!=0.19.0,<=0.24.0,>=0.18.2; sys_platform != "darwin" or platform_machine != "arm64", which is not installed.
unsloth 2026.5.7 requires bitsandbytes!=0.46.0,!=0.48.0,>=0.45.5, which is not installed.
unsloth 2026.5.7 requires trl!=0.19.0,<=0.24.0,>=0.18.2, which is not installed.
unsloth-zoo 2026.5.4 requires peft!=0.11.0,>=0.18.0; sys_platform != "darwin" or

In [ ]:
import shutil

shutil.make_archive(
    "/content/unsloth-qwen-bible",
    'zip',
    "/content/unsloth-qwen-bible"
)

'/content/unsloth-qwen-bible.zip'

In [ ]:
from google.colab import files

files.download("/content/unsloth-qwen-bible.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
model.push_to_hub("ngandu2772/mon-modele-bible")
tokenizer.push_to_hub("ngandu2772/mon-modele-bible")

HfHubHTTPError: (Request ID: Root=1-6a137abb-562616410ce8e5a07cee9e92;97e78ce4-ea7f-4718-abf4-9faeb2829e31)

403 Forbidden: You don't have the rights to create a model under the namespace "ngandu2772".
Cannot access content at: https://huggingface.co/api/repos/create.
Make sure your token has the correct permissions.